In [1]:
import os
import time
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoAlertPresentException
from webdriver_manager.chrome import ChromeDriverManager

In [2]:
# ===============================
# 1. 기본 설정
# ===============================
TARGET_URL = "https://www.mnd.go.kr/cop/pblictn/selectPublicationsUser.do?siteId=mnd&componentId=14&categoryId=18&pageIndex=1&id=mnd_020704000000"
SAVE_DIR = os.path.abspath("국방부_정책자료")

os.makedirs(SAVE_DIR, exist_ok=True)

chrome_options = Options()
prefs = {
    "download.default_directory": SAVE_DIR,
    "download.prompt_for_download": False,
    "directory_upgrade": True,
    "plugins.always_open_pdf_externally": True
}
chrome_options.add_experimental_option("prefs", prefs)
chrome_options.add_argument("--ignore-certificate-errors")
chrome_options.add_argument("--ignore-ssl-errors")

driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=chrome_options
)

wait = WebDriverWait(driver, 15)

# ===============================
# 2. 다운로드 로직
# ===============================
def start_download(max_page=3):  # ← 페이지 수 조절 가능
    driver.get(TARGET_URL)

    for page in range(1, max_page + 1):
        print(f"\n📄 페이지 {page} 처리 중...")

        # 페이지 이동 (1페이지는 최초 로딩)
        if page > 1:
            driver.execute_script(f"publicationSearch('{page}')")
            time.sleep(0.5)

        # 게시글 리스트 로딩 대기
        wait.until(EC.presence_of_element_located(
            (By.CSS_SELECTOR, "ul.list_post > li")
        ))

        posts = driver.find_elements(By.CSS_SELECTOR, "ul.list_post > li")
        print(f"   ▶ 게시글 수: {len(posts)}")

        for i in range(len(posts)):
            try:
                # DOM 재로딩 대응
                posts = driver.find_elements(By.CSS_SELECTOR, "ul.list_post > li")
                post = posts[i]

                link = post.find_element(By.CSS_SELECTOR, "a")
                title = link.text.strip() or f"Unknown_{page}_{i}"

                print(f"   [{i+1}] {title}")

                # 상세 페이지 이동
                driver.execute_script("arguments[0].click();", link)
                time.sleep(0.5)

                # 다운로드 링크 탐색 (PDF)
                download_links = driver.find_elements(
                    By.CSS_SELECTOR,
                    "a[href*='.pdf'], a[href*='download']"
                )

                if not download_links:
                    print("      ❌ PDF 없음")
                    driver.back()
                    time.sleep(0.2)
                    continue

                for dl in download_links:
                    try:
                        print("      ⬇ PDF 다운로드")
                        driver.execute_script("arguments[0].click();", dl)
                        time.sleep(1)

                        # 알림창 처리
                        try:
                            alert = driver.switch_to.alert
                            alert.accept()
                        except NoAlertPresentException:
                            pass

                    except Exception as e:
                        print(f"      ❌ 다운로드 오류: {e}")

                driver.back()
                time.sleep(0.5)

            except Exception as e:
                print(f"   ❌ 게시글 처리 오류: {e}")
                driver.back()
                time.sleep(0.5)

    print(f"\n✅ 다운로드 완료: {SAVE_DIR}")
    time.sleep(1)
    driver.quit()


if __name__ == "__main__":
    start_download(max_page=3)


📄 페이지 1 처리 중...
   ▶ 게시글 수: 5
   [1] Unknown_1_0
      ⬇ PDF 다운로드
      ⬇ PDF 다운로드
      ⬇ PDF 다운로드
   [2] Unknown_1_1
      ❌ PDF 없음
   [3] Unknown_1_2
      ⬇ PDF 다운로드
   [4] Unknown_1_3
      ⬇ PDF 다운로드
   [5] Unknown_1_4
      ⬇ PDF 다운로드

📄 페이지 2 처리 중...
   ▶ 게시글 수: 1
   [1] Unknown_2_0
      ⬇ PDF 다운로드

📄 페이지 3 처리 중...
   ▶ 게시글 수: 1
   ❌ 게시글 처리 오류: Message: no such element: Unable to locate element: {"method":"css selector","selector":"a"}
  (Session info: chrome=143.0.7499.170); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#nosuchelementexception
Stacktrace:
Symbols not available. Dumping unresolved backtrace:
	0x5d12d3
	0x5d1314
	0x3be6dd
	0x4093a5
	0x40977b
	0x3fed31
	0x42b534
	0x3fec34
	0x42b6a4
	0x44db13
	0x42b2e6
	0x3fd321
	0x3fe1d4
	0x825314
	0x8208cb
	0x83d1aa
	0x5eb1d8
	0x5f31dd
	0x5d95d8
	0x5d9799
	0x5c3b28
	0x75c55d49
	0x772ad5db
	0x772ad561


✅ 다운로드 완료: c:\Users\user\OneDrive\Desktop\Desktop\VSCo

한 번에 처리 안될 때
- 이미지 단위로 다운받아서 묶어서 pdf화

In [3]:
! pip install pillow requests selenium webdriver-manager


[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time
import re
from selenium import webdriver
import os
import requests
from PIL import Image

In [11]:
driver = webdriver.Chrome()
driver.get("https://www.mnd.go.kr/mbshome/mbs/reform/reform_ebook/reform2.0/index.html")

WebDriverWait(driver, 15).until(EC.presence_of_element_located((By.TAG_NAME, "body")))
time.sleep(2)

page_image_urls = set()  # 중복 제거를 위해 set 유지
TOTAL_PAGES = 64
body = driver.find_element(By.TAG_NAME, "body")

print("📸 이미지 URL 수집 시작...")

for page in range(TOTAL_PAGES):
    time.sleep(0.7)  # 로딩 대기 시간 살짝 증가
    imgs = driver.find_elements(By.TAG_NAME, "img")
    
    for img in imgs:
        src = img.get_attribute("src")
        # 실제 고화질 페이지 이미지만 필터링
        if src and "page-" in src and not src.endswith("-thumb.jpg"):
            page_image_urls.add(src)

    body.send_keys(Keys.ARROW_RIGHT)
    if page % 10 == 0:
        print(f"진행 중: {page}/{TOTAL_PAGES} 완료")

driver.quit()

📸 이미지 URL 수집 시작...
진행 중: 0/64 완료
진행 중: 10/64 완료
진행 중: 20/64 완료
진행 중: 30/64 완료
진행 중: 40/64 완료
진행 중: 50/64 완료
진행 중: 60/64 완료


In [12]:
page_items = []
for url in page_image_urls:
    # URL에서 숫자를 찾아 정수로 변환 (예: page-012.jpg -> 12)
    import re
    match = re.search(r'page-(\d+)', url)
    if match:
        page_no = int(match.group(1))
        page_items.append((page_no, url))

# 실제 페이지 번호 순서대로 정렬
page_items.sort(key=lambda x: x[0])

print(f"✅ 총 {len(page_items)}개의 페이지를 순서대로 정리했습니다.")

✅ 총 64개의 페이지를 순서대로 정리했습니다.


In [14]:
IMAGE_DIR = "ebook_images"
os.makedirs(IMAGE_DIR, exist_ok=True)

# 헤더 설정 (국방부 사이트 보안 대응)
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Referer": "https://www.mnd.go.kr/mbshome/mbs/reform/reform_ebook/reform2.0/index.html"
}

print(f"🚀 다운로드를 시작합니다. (총 {len(page_items)}개)")

# page_items는 (페이지번호, URL) 형태로 정렬되어 있어야 함
for page_no, url in page_items:
    # 파일명을 001.jpg, 002.jpg 형태로 고정 (순서 정렬의 핵심)
    filename = f"{page}_{page_no}.jpg"
    path = os.path.join(IMAGE_DIR, filename)

    try:
        r = requests.get(url, headers=headers, timeout=15)
        
        if r.status_code == 200:
            with open(path, "wb") as f:
                f.write(r.content)
            print(f"✅ [저장 완료] {filename}")
        else:
            print(f"❌ [다운 실패] {filename} (상태코드: {r.status_code})")
            
    except Exception as e:
        print(f"⚠️ [오류 발생] {filename}: {e}")

print("\n✨ 모든 페이지 다운로드가 완료되었습니다!")
print(f"저장 위치: {os.path.abspath(IMAGE_DIR)}")

🚀 다운로드를 시작합니다. (총 64개)
✅ [저장 완료] 63_1352110000.jpg
✅ [저장 완료] 63_1352110000.jpg
✅ [저장 완료] 63_1352110000.jpg
✅ [저장 완료] 63_1352110000.jpg
✅ [저장 완료] 63_1352110000.jpg
✅ [저장 완료] 63_1352110000.jpg
✅ [저장 완료] 63_1352110000.jpg
✅ [저장 완료] 63_1352110000.jpg
✅ [저장 완료] 63_1352110000.jpg
✅ [저장 완료] 63_1352110000.jpg
✅ [저장 완료] 63_1352110000.jpg
✅ [저장 완료] 63_1352110000.jpg
✅ [저장 완료] 63_1352110000.jpg
✅ [저장 완료] 63_1352110000.jpg
✅ [저장 완료] 63_1352110000.jpg
✅ [저장 완료] 63_1352110000.jpg
✅ [저장 완료] 63_1352110000.jpg
✅ [저장 완료] 63_1352110000.jpg
✅ [저장 완료] 63_1352110000.jpg
✅ [저장 완료] 63_1352110000.jpg
✅ [저장 완료] 63_1352110000.jpg
✅ [저장 완료] 63_1352110000.jpg
✅ [저장 완료] 63_1352110000.jpg
✅ [저장 완료] 63_1352110000.jpg
✅ [저장 완료] 63_1352110000.jpg
✅ [저장 완료] 63_1352110000.jpg
✅ [저장 완료] 63_1352110000.jpg
✅ [저장 완료] 63_1352110000.jpg
✅ [저장 완료] 63_1352110000.jpg
✅ [저장 완료] 63_1352110000.jpg
✅ [저장 완료] 63_1352110000.jpg
✅ [저장 완료] 63_1352110000.jpg
✅ [저장 완료] 63_1352110000.jpg
✅ [저장 완료] 63_1352110000.jpg
✅ [저장 완료] 63_1352110000.j

In [ ]:
if not image_files:
    raise RuntimeError("❌ 저장된 이미지가 없습니다")

images = [Image.open(f).convert("RGB") for f in image_files]

pdf_path = "국방개혁_2.0.pdf"
images[0].save(pdf_path, save_all=True, append_images=images[1:])

print(f"✅ PDF 생성 완료: {pdf_path}")

한 번에 다 할 때 아래 코드 실행

In [25]:
import os
import time
from PIL import Image
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# ================== 설정 ==================
IMAGE_DIR = "ebook_images"
os.makedirs(IMAGE_DIR, exist_ok=True)

TOTAL_PAGES = 64
PDF_PATH = "국방개혁_2.0.pdf"

URL = "https://www.mnd.go.kr/mbshome/mbs/reform/reform_ebook/reform2.0/index.html"
# =========================================

driver = webdriver.Chrome()
driver.get(URL)

WebDriverWait(driver, 15).until(
    EC.presence_of_element_located((By.TAG_NAME, "body"))
)
time.sleep(2)

body = driver.find_element(By.TAG_NAME, "body")

print("📸 페이지 캡처 시작")

image_paths = []

for page in range(1, TOTAL_PAGES + 1):
    filename = f"{page:03}.png"
    path = os.path.join(IMAGE_DIR, filename)

    # 화면 캡처 (현재 페이지)
    driver.save_screenshot(path)
    image_paths.append(path)

    print(f"✅ {page}페이지 캡처 완료")

    # 마지막 페이지가 아니면 다음 페이지로
    if page != TOTAL_PAGES:
        body.send_keys(Keys.ARROW_RIGHT)
        time.sleep(0.6)  # 페이지 렌더링 대기

driver.quit()

print("📄 PDF 생성 중...")

# PNG → PDF
images = [Image.open(p).convert("RGB") for p in image_paths]
images[0].save(
    PDF_PATH,
    save_all=True,
    append_images=images[1:]
)

print(f"✨ 완료! PDF 생성됨: {PDF_PATH}")

📸 페이지 캡처 시작
✅ 1페이지 캡처 완료
✅ 2페이지 캡처 완료
✅ 3페이지 캡처 완료
✅ 4페이지 캡처 완료
✅ 5페이지 캡처 완료
✅ 6페이지 캡처 완료
✅ 7페이지 캡처 완료
✅ 8페이지 캡처 완료
✅ 9페이지 캡처 완료
✅ 10페이지 캡처 완료
✅ 11페이지 캡처 완료
✅ 12페이지 캡처 완료
✅ 13페이지 캡처 완료
✅ 14페이지 캡처 완료
✅ 15페이지 캡처 완료
✅ 16페이지 캡처 완료
✅ 17페이지 캡처 완료
✅ 18페이지 캡처 완료
✅ 19페이지 캡처 완료
✅ 20페이지 캡처 완료
✅ 21페이지 캡처 완료
✅ 22페이지 캡처 완료
✅ 23페이지 캡처 완료
✅ 24페이지 캡처 완료
✅ 25페이지 캡처 완료
✅ 26페이지 캡처 완료
✅ 27페이지 캡처 완료
✅ 28페이지 캡처 완료
✅ 29페이지 캡처 완료
✅ 30페이지 캡처 완료
✅ 31페이지 캡처 완료
✅ 32페이지 캡처 완료
✅ 33페이지 캡처 완료
✅ 34페이지 캡처 완료
✅ 35페이지 캡처 완료
✅ 36페이지 캡처 완료
✅ 37페이지 캡처 완료
✅ 38페이지 캡처 완료
✅ 39페이지 캡처 완료
✅ 40페이지 캡처 완료
✅ 41페이지 캡처 완료
✅ 42페이지 캡처 완료
✅ 43페이지 캡처 완료
✅ 44페이지 캡처 완료
✅ 45페이지 캡처 완료
✅ 46페이지 캡처 완료
✅ 47페이지 캡처 완료
✅ 48페이지 캡처 완료
✅ 49페이지 캡처 완료
✅ 50페이지 캡처 완료
✅ 51페이지 캡처 완료
✅ 52페이지 캡처 완료
✅ 53페이지 캡처 완료
✅ 54페이지 캡처 완료
✅ 55페이지 캡처 완료
✅ 56페이지 캡처 완료
✅ 57페이지 캡처 완료
✅ 58페이지 캡처 완료
✅ 59페이지 캡처 완료
✅ 60페이지 캡처 완료
✅ 61페이지 캡처 완료
✅ 62페이지 캡처 완료
✅ 63페이지 캡처 완료
✅ 64페이지 캡처 완료
📄 PDF 생성 중...
✨ 완료! PDF 생성됨: 국방개혁_2.0.pdf
